# Desktop Scene Reconstruction — CP260 Final Project
**Pipeline:** plane-sweep depth → OWL-ViT + MobileSAM → RANSAC plane-fit OBB → image-based view synthesis.

This notebook is a thin wrapper around the modules under `src/`.  Each stage of the pipeline lives in its own module so the code can also be run as a script (`python -m src.driver --sam-ckpt mobile_sam.pt`).

## 0 · Environment check

In [ ]:
import torch, sys, platform
print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())
print('CUDA     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))
!nvidia-smi -L

## 1 · Install dependencies

In [ ]:
%%capture
!pip install -q numpy opencv-python matplotlib tqdm pillow
!pip install -q transformers==4.41.2 accelerate
!pip install -q timm
# MobileSAM — small Segment-Anything variant
!pip install -q git+https://github.com/ChaoningZhang/MobileSAM.git

In [ ]:
# Download the MobileSAM weights (~40 MB)
!wget -q https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt -O mobile_sam.pt
!ls -lh mobile_sam.pt

## 2 · Upload data
Upload the **Data.zip** that the professor shared.

In [ ]:
from google.colab import files
_ = files.upload()  # pick Data.zip

In [ ]:
!unzip -q -o Data.zip -d .
!ls Data | head -20
import json, os
with open('Data/poses.json') as fh:
    poses_raw = json.load(fh)
frames = sorted([int(f.replace('frame_','').replace('.png','')) for f in os.listdir('Data') if f.endswith('.png')])
print('Image frames :', frames)
print('Poses keys   :', len(poses_raw))

In [ ]:
# Drop in our intrinsic.json (values from the previous course release)
intrinsic = {
    'camera_matrix': [
        [1477.00974684544, 0.0, 1298.2501500778505],
        [0.0, 1480.4424455584467, 686.8201623541711],
        [0.0, 0.0, 1.0]
    ],
    'image_width': 2560,
    'image_height': 1440
}
with open('intrinsic.json', 'w') as fh:
    json.dump(intrinsic, fh, indent=2)

## 3 · Pull the source modules
Either upload a zip of the `src/` folder (next cell) **or** clone your own GitHub repo.

In [ ]:
# Option A: upload a zip containing src/ (recommended while iterating)
from google.colab import files
print('Upload src.zip (or skip this cell if you cloned a git repo)')
_ = files.upload()
!unzip -q -o src.zip -d .
!ls src/

## 4 · Quick sanity-check on the data

In [ ]:
import importlib, sys
sys.path.insert(0, '.')
from src import asset_io, settings as S
importlib.reload(asset_io)

images, poses, K = asset_io.load_full_capture(resize=1.0)
print(f'Loaded {len(images)} images, {len(poses)} poses')
print('K =\n', K)

In [ ]:
import matplotlib.pyplot as plt
ids = sorted(images.keys())[:8]
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.flat, ids):
    ax.imshow(images[fid]); ax.set_title(f'frame {fid}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 5 · Run the full pipeline

In [ ]:
# ⚠️ run with --fast first to check that everything works (≈3 min)
from src import driver, settings as S
import importlib
for mod_name in ['settings','asset_io','depth_sweep','vlm_segmenter','box_fitter','view_render','render_helpers','driver']:
    importlib.reload(importlib.import_module(f'src.{mod_name}'))
from src import driver

result = driver.run(sam_ckpt='mobile_sam.pt', fast=True)
records = result['records']
depth_maps = result['depth_maps']
conf_maps = result['conf_maps']
images = result['images']; poses = result['poses']; K = result['K']

In [ ]:
# Production run at full resolution
result = driver.run(sam_ckpt='mobile_sam.pt', fast=False)
records = result['records']
depth_maps = result['depth_maps']
conf_maps = result['conf_maps']
images = result['images']; poses = result['poses']; K = result['K']

## 6 · Inspect the depth maps

In [ ]:
import glob, cv2
previews = sorted(glob.glob('outputs/viz/depth_*.png'))[:8]
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, p in zip(axes.flat, previews):
    ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    ax.set_title(p.split('/')[-1]); ax.axis('off')
plt.tight_layout(); plt.show()

## 7 · OBB overlay on the source frames

In [ ]:
import json, numpy as np, cv2
from src import render_helpers, asset_io

with open('outputs/answers.json') as fh:
    answers = json.load(fh)

palette = [(255, 60, 60), (60, 200, 80), (255, 165, 0), (180, 90, 255)]
for fid in [471, 496]:
    if fid not in poses: continue
    img = (images[fid] * 255).astype(np.uint8).copy()
    for i, rec in enumerate(answers):
        obb = rec['obb']
        corners = render_helpers.obb_corners(np.array(obb['center']),
                                             np.array(obb['extent']),
                                             np.array(obb['rotation']))
        proj = render_helpers.project_corners(corners, K, poses[fid])
        if proj is None: continue
        img = render_helpers.draw_box_overlay(img, proj,
                                              colour=palette[i % len(palette)],
                                              label=rec['entity'])
    plt.figure(figsize=(11, 6)); plt.imshow(img); plt.axis('off')
    plt.title(f'OBBs projected onto frame {fid}'); plt.show()

## 8 · 3-D scene plot

In [ ]:
render_helpers.plot_scene_3d(poses, answers,
                             save_path='outputs/viz/scene3d.png')

## 9 · Bonus: detect additional connectors
If the professor asks for a new entity at evaluation time, add a prompt and re-run only the segmentation+OBB stages.

In [ ]:
# Example: add an `audio_jack` entity on the fly
extra_prompts = {
    'audio_jack':           'a green or pink 3.5mm audio jack on a computer back panel',
    'usb_socket_top_right': 'the topmost USB-A port on a computer back panel',
}
from src import vlm_segmenter, box_fitter, depth_sweep
extra_hits = vlm_segmenter.segment_all(images, 'mobile_sam.pt', prompts=extra_prompts)
# Recompute depth maps for the frames where the new entity was detected
needed_frames = {h.frame_id for hits in extra_hits.values() for h in hits}
depth_maps_extra, conf_maps_extra = {}, {}
for fid in needed_frames:
    d, c = depth_sweep.estimate_depth_map(fid, images, poses, K, n_levels=64, progress=False)
    depth_maps_extra[fid] = d; conf_maps_extra[fid] = c
extra_records = box_fitter.fit_all_boxes(extra_hits, depth_maps_extra, conf_maps_extra, K, poses)
print('extra OBBs:', extra_records)

## 10 · Download outputs

In [ ]:
!cd outputs && zip -qr ../outputs.zip . && cd ..
from google.colab import files
files.download('outputs.zip')